In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

# Executive Summary

### Tuning performance of *Humanoid-v5*.

We first look at PDA

In [ ]:
path = "/Users/calebju/Code/RL-general-action-state/logs"
n_seeds = 1

# 02_12_2026/exp_0.py 
data_1 = np.zeros(24, dtype=float)
for i in range(len(data_1)):
    df = pd.read_csv(os.path.join(path, "02_12_2026/exp_0/run_%d/seed=0.csv" % (i,)))
    data_1[i] = df['episode rewards'].iloc[-1]

In [ ]:
for i in range(len(data_1)):
    print("%d: %.4e %s" % (i, -data_1[i], "*" if -data_1[i] == np.min(-data_1) else ""))

Next, we do PPO and DDPG.

In [ ]:
path = "/Users/calebju/Code/RL-general-action-state/logs"
n_seeds = 1

# 02_12_2026/exp_2.py 
data_1 = np.zeros(14, dtype=float)
for i in range(len(data_1)):
    df = pd.read_csv(os.path.join(path, "02_12_2026/exp_2/run_%d/seed=0.csv" % (i,)))
    data_1[i] = df['episode rewards'].iloc[-1]

In [ ]:
print("Tuning PPO\n")
for i in range(7):
    print("%d: %.4e %s" % (i, -data_1[i], "*" if -data_1[i] == np.min(-data_1[0:7]) else ""))

print("\nTuning DDPG\n")
for i in range(7,14):
    print("%d: %.4e %s" % (i, -data_1[i], "*" if -data_1[i] == np.min(-data_1[7:14]) else ""))

This corresponds to `ppo_lr=0.01` and `ddpg=0.0003`.

### Humanoid performance

Note that we do not run tuned DDPG since we got the error:
```
UserWarning: This system does not have apparently enough memory to store the complete replay buffer 5.64GB > 5.55GB
```

In [ ]:
path = "/Users/calebju/Code/RL-general-action-state/logs"
n_seeds = 5

# 02_12_2026/exp_0.py 
data_2 = np.zeros((4,n_seeds,500_000), dtype=float)
aux_2 = np.zeros((4,n_seeds,500_000), dtype=float)
len_2 = np.zeros((4, n_seeds), dtype=int)
for i in range(len(data_2)):
    for j in range(n_seeds):
        df = pd.read_csv(os.path.join(path, "02_12_2026/exp_1/run_%d/seed=%d.csv" % (i, j)))
        temp = df['episode rewards']
        temp2 = df['episode len']
        len_2[i,j] = len(temp)
        # data_1[i,j,:len_1[i,j]] = np.convolve(temp, ones_5)[:-2]
        data_2[i,j,:len_2[i,j]] = temp
        aux_2[i,j,:len_2[i,j]] = np.cumsum(temp2)

In [ ]:
plt.style.use('ggplot')
_, ax = plt.subplots(figsize=(10,7))

label_arr = ['pda', 'ppo', 'ddpg', 'ppo (Zoo)']
lss_arr = ['solid', 'dashed', 'dotted', "dashdot"]
color_arr = ['red', 'green', 'purple', 'blue']
ones_5 = 0.05*np.ones(20)

for i in range(len(data_2)):
    l = np.min(len_2[i])
    xs = np.max(aux_2[i,:,:l], axis=0)[::5]
    ys = np.max(data_2[i,:,:l], axis=0)[::5]
    ys = np.convolve(ys, ones_5)[:-len(ones_5)+1]
    # rng = np.std(data_1[i,:,:l], axis=0)[::5]
    ax.plot(xs, -ys, label=label_arr[i], linestyle=lss_arr[i], color=color_arr[i])
    # ax.fill_between(xs, -ys-rng, -ys+rng, color=color_arr[i], alpha=0.1)

ax.legend()
ax.set(
    title="Costs in Humanoid over %d trials" % n_seeds,
    ylabel="Smoothed cumulative cost\n(lower is better)",
    xlabel="Samples",
    xlim=(-5_000, 100_000),
    ylim=(-4e2, 1e2)
)

Similar to `2026_02_11.ipynb`, we will bucket the lengths.

In [ ]:
xs = np.append(
    np.append(np.arange(1_000, step=10), np.arange(1_000, 10_000, step=25)), 
    np.arange(10_000, 100_000, step=100)
)
clean_2 = np.zeros((data_2.shape[0], n_seeds, len(xs)), dtype=int)
for i in range(clean_2.shape[0]):
    for j in range(clean_2.shape[1]):
        for k,x in enumerate(xs):
            # finds index i such that len_i < x <= len_(i+1)
            clean_2[i,j,k] = np.argmax(x <= aux_2[i,j,:])

In [ ]:
plt.style.use('ggplot')
_, ax = plt.subplots(figsize=(5,5))

for i in range(len(data_2)):
    ys = np.zeros(clean_2.shape[1:])
    for j in range(n_seeds):
        ys[j] = data_2[i, j, clean_2[i,j]]
        ys[j] = np.convolve(ys[j], ones_5)[:-len(ones_5)+1]
    med = np.mean(ys, axis=0)
    rng = np.std(ys, axis=0)
    rng *= 2.571 # based on 2-sided t-score with p=0.05
    ax.plot(xs, -med, label=label_arr[i], linestyle=lss_arr[i], color=color_arr[i])
    ax.fill_between(xs, -med-rng, -med+rng, color=color_arr[i], alpha=0.1)

ax.legend(loc="lower left")
ax.set(
    title="Costs in Humanoid over 5 trials",
    ylabel="Cumulative discounted cost\n smoothed over %d periods" % len(ones_5),
    xlabel="Samples",
    ylim=(-400,50),
    xlim=(-2_000, 100_000),
)

plt.tight_layout()
# plt.savefig("humanoid.png", dpi=90)
plt.savefig("humanoid.png", dpi=120)

It looks lilke DDPG performs the best followed by PDA. But the variance of PPO is the largest. Let us plot it.

In [ ]:
plt.style.use('ggplot')
_, axes = plt.subplots(ncols=2, figsize=(8,5))
method_arr = ['pda', 'ddpg']

i_s = [0,2]
ones_5 = np.ones(100)
ones_5 /= len(ones_5)

for k,i in enumerate(i_s):
    ys = np.zeros(aux_2.shape[1:])
    for j in range(n_seeds):
        l = len_2[i,j]
        _xs = aux_2[i,j,:l:]
        ys = np.convolve(data_2[i,j,:l:], ones_5)[:-len(ones_5)+1]
        axes[k].plot(_xs, -ys, label=label_arr[i])

    # ax.legend(loc="right")
    axes[k].set(
        title="%s on Humanoid" % method_arr[k],
        xlim=(-1_000, 100_000),
        ylim=(-325,25),
        xlabel="Samples",
    )
    
# ax.legend(loc="right")
axes[0].set(
    title="pda on Humanoid",
    ylabel="Cumulative discounted cost\n smoothed over %d periods" % len(ones_5),
    xlabel="Samples",
    xlim=(-1_000, 100_000),
    ylim=(-325,25),
)
axes[1].tick_params(labelleft=False)   

# plt.suptitle("Costs in Humanoid over %d different seeds (pda left, ddpg right)" % (n_seeds))

plt.tight_layout()
# plt.savefig("humanoid_seeds.png", dpi=90)
plt.savefig("humanoid_seeds.png", dpi=120)